In [22]:
# 식품 영양 정보를 가져와서 csv 파일로 저장
# 가져올 정보: 식품명, 탄수화물, 지방, 단백질, 나트륨, 총칼로리

In [23]:
import pandas as pd
import requests

In [ ]:
# 공공데이터 url = End Point + 상세기능 url
url = 'https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02'

serviceKey = '9f45f7a1696e595dddf89a923c94381ac985eb9256f89b343fb93bb0477d020b'
page = 1

params = {
	'serviceKey' : serviceKey, # 필수, 나머지는 선택
	'pageNo' : page,
	'numOfRows' : 10,
	'type' : 'json',
}

response = requests.get(url, params = params)

try:
	response.raise_for_status()

	print(response.json)
except Exception as e:
	print(f"예외 발생 : {e}")



<bound method Response.json of <Response [200]>>


In [25]:
items = response.json()['body']['items']

list = []
for item in items:
	# 식품명, 탄수활물, 지방, 단백질, 나트륨, 총칼로리
	dic = {
		'식품명' : item['FOOD_NM_KR'],
		'탄수화물(g)' : item['AMT_NUM6'],
		'지방' : item['AMT_NUM4'],
		'단백질': item['AMT_NUM3'],
		'나트륨' : item['AMT_NUM13'],
		'총칼로리' : item['AMT_NUM1'],
		'기준량' : item['SERVING_SIZE'],
	}
	list.append(dic)
	print(dic)

	# 받아온 리스트를 데이터프레임으로 변환
	df = pd.DataFrame(list)


{'식품명': '국밥_돼지머리', '탄수화물(g)': '15.94', '지방': '5.16', '단백질': '6.70', '나트륨': '181.000', '총칼로리': '137.000', '기준량': '100g'}
{'식품명': '국밥_순대국밥', '탄수화물(g)': '10.38', '지방': '2.28', '단백질': '3.17', '나트륨': '126.000', '총칼로리': '75.00', '기준량': '100g'}
{'식품명': '국밥_콩나물', '탄수화물(g)': '10.93', '지방': '0.24', '단백질': '1.45', '나트륨': '172.000', '총칼로리': '52.00', '기준량': '100g'}
{'식품명': '기장밥', '탄수화물(g)': '36.77', '지방': '0.57', '단백질': '3.44', '나트륨': '1.00', '총칼로리': '166.000', '기준량': '100g'}
{'식품명': '김밥', '탄수화물(g)': '19.98', '지방': '4.55', '단백질': '4.84', '나트륨': '307.000', '총칼로리': '140.000', '기준량': '100g'}
{'식품명': '김밥_김치', '탄수화물(g)': '19.17', '지방': '4.03', '단백질': '4.30', '나트륨': '349.000', '총칼로리': '130.000', '기준량': '100g'}
{'식품명': '김밥_날치알', '탄수화물(g)': '28.66', '지방': '4.26', '단백질': '6.10', '나트륨': '299.000', '총칼로리': '177.000', '기준량': '100g'}
{'식품명': '김밥_돈가스', '탄수화물(g)': '31.64', '지방': '5.81', '단백질': '5.77', '나트륨': '241.000', '총칼로리': '202.000', '기준량': '100g'}
{'식품명': '김밥_소고기', '탄수화물(g)': '25.78', '지방': '5.56', '단백질': '6

In [26]:
# 중복 제거
# df.drop_duplicates() # 행 전체 값이 중복되면 제거
# df.drop_duplicates(subset=['식품명']) # 식품명 열이 중복된 경우만 제거
df = df.drop_duplicates(subset=['식품명'], keep='first') # 중복되면 첫번째를 살리고 나머지 제거
# df.drop_duplicates(subset=['식품명'], keep='last') # 중복되면 마지막을 살리고 나머지 제거
# df.drop_duplicates(subset=['식품명'], keep=False) # 중복되면 모두 제거
print(df)
#  결측치 값 변경
df['탄수화물(g)'].fillna(0) # 탄수화물(g)에 결측치가 있으면 0으로 변경

# 결측치 제거
df = df.dropna() # 행에 결측치가 하나라도 있으면 삭제
df = df.dropna(subset=['식품명']) # 식품명에 결측치가 있으면 삭제

       식품명 탄수화물(g)    지방   단백질      나트륨     총칼로리   기준량
0  국밥_돼지머리   15.94  5.16  6.70  181.000  137.000  100g
1  국밥_순대국밥   10.38  2.28  3.17  126.000    75.00  100g
2   국밥_콩나물   10.93  0.24  1.45  172.000    52.00  100g
3      기장밥   36.77  0.57  3.44     1.00  166.000  100g
4       김밥   19.98  4.55  4.84  307.000  140.000  100g
5    김밥_김치   19.17  4.03  4.30  349.000  130.000  100g
6   김밥_날치알   28.66  4.26  6.10  299.000  177.000  100g
7   김밥_돈가스   31.64  5.81  5.77  241.000  202.000  100g
8   김밥_소고기   25.78  5.56  6.46  267.000  179.000  100g
9    김밥_참치   20.26  7.22  7.00  335.000  174.000  100g


In [ ]:
def get_food_nutrient(serviceKey: str, page:int=1)->pd.DataFrame:
	"""공공데이터에서 식품영양소 db 정보를 가져와서 데이터 프레임으로 반환하는 함수"""

	url='https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02'
	serviceKey = '9f45f7a1696e595dddf89a923c94381ac985eb9256f89b343fb93bb0477d020b'
	params = {
		'ServiceKey' : serviceKey,
		'pageNo' : page,
		'numOfRows': 10,
    'type': 'json'
	}

	try:
		response = requests.get(url, params=params)
		response.raise_for_status()

		items = response.json()['body']['items']
		list = []

		labels = {
			'식품명' : 'FOOD_NM_KR',
			'탄수화물' : 'AMT_NUM6',
			'지방' : 'AMT_NUM4',
			'단백질' : 'AMT_NUM3',
			'나트륨' : 'AMT_NUM13',
			'총칼로리' : 'AMT_NUM1',
			'기준량' : 'SERVING_SIZE'
		}

		for item in items:
			dic ={
				'식품명' : item['FOOD_NM_KR'],
				'탄수화물(g)' : item['AMT_NUM6'],
				'지방' : item['AMT_NUM4'],
				'단백질': item['AMT_NUM3'],
				'나트륨' : item['AMT_NUM13'],
				'총칼로리' : item['AMT_NUM1'],
				'기준량' : item['SERVING_SIZE'],
			}
			list.append(dic)
			print(list)
			df = pd.DataFrame(list)
			print(df)

	except Exception as e:
		print(f"에러 발생 : {e}")

get_food_nutrient('9f45f7a1696e595dddf89a923c94381ac985eb9256f89b343fb93bb0477d020b', 2)
		

[{'식품명': '김밥_채소', '탄수화물(g)': '26.65', '지방': '3.65', '단백질': '4.60', '나트륨': '309.000', '총칼로리': '158.000', '기준량': '100g'}]
     식품명 탄수화물(g)    지방   단백질      나트륨     총칼로리   기준량
0  김밥_채소   26.65  3.65  4.60  309.000  158.000  100g
[{'식품명': '김밥_채소', '탄수화물(g)': '26.65', '지방': '3.65', '단백질': '4.60', '나트륨': '309.000', '총칼로리': '158.000', '기준량': '100g'}, {'식품명': '김밥_치즈', '탄수화물(g)': '22.10', '지방': '7.03', '단백질': '6.24', '나트륨': '169.000', '총칼로리': '177.000', '기준량': '100g'}]
     식품명 탄수화물(g)    지방   단백질      나트륨     총칼로리   기준량
0  김밥_채소   26.65  3.65  4.60  309.000  158.000  100g
1  김밥_치즈   22.10  7.03  6.24  169.000  177.000  100g
[{'식품명': '김밥_채소', '탄수화물(g)': '26.65', '지방': '3.65', '단백질': '4.60', '나트륨': '309.000', '총칼로리': '158.000', '기준량': '100g'}, {'식품명': '김밥_치즈', '탄수화물(g)': '22.10', '지방': '7.03', '단백질': '6.24', '나트륨': '169.000', '총칼로리': '177.000', '기준량': '100g'}, {'식품명': '김밥_풋고추', '탄수화물(g)': '27.52', '지방': '4.41', '단백질': '4.88', '나트륨': '327.000', '총칼로리': '169.000', '기준량': '100g'}]
      식품명 탄수화물(g)

In [ ]:
def clean_food_nutrient(datas:pd.DataFrame)->pd.DataFrame:
	"""식품영양소 df에서 결측치 제거, 중복 제거하는 함수"""
	df = df.drop_duplicates(subset=['식품명'], keep='first')

	#  결측치 값 변경
	df['탄수화물(g)'].fillna(0) # 탄수화물(g)에 결측치가 있으면 0으로 변경

	# 결측치 제거
	df = df.dropna() # 행에 결측치가 하나라도 있으면 삭제
	df = df.dropna(subset=['식품명']) # 식품명에 결측치가 있으면 삭제

In [40]:
food_df = get_food_nutrient(serviceKey, 1)
food_df = clean_food_nutrient(food_df)


[{'식품명': '국밥_돼지머리', '탄수화물(g)': '15.94', '지방': '5.16', '단백질': '6.70', '나트륨': '181.000', '총칼로리': '137.000', '기준량': '100g'}]
       식품명 탄수화물(g)    지방   단백질      나트륨     총칼로리   기준량
0  국밥_돼지머리   15.94  5.16  6.70  181.000  137.000  100g
[{'식품명': '국밥_돼지머리', '탄수화물(g)': '15.94', '지방': '5.16', '단백질': '6.70', '나트륨': '181.000', '총칼로리': '137.000', '기준량': '100g'}, {'식품명': '국밥_순대국밥', '탄수화물(g)': '10.38', '지방': '2.28', '단백질': '3.17', '나트륨': '126.000', '총칼로리': '75.00', '기준량': '100g'}]
       식품명 탄수화물(g)    지방   단백질      나트륨     총칼로리   기준량
0  국밥_돼지머리   15.94  5.16  6.70  181.000  137.000  100g
1  국밥_순대국밥   10.38  2.28  3.17  126.000    75.00  100g
[{'식품명': '국밥_돼지머리', '탄수화물(g)': '15.94', '지방': '5.16', '단백질': '6.70', '나트륨': '181.000', '총칼로리': '137.000', '기준량': '100g'}, {'식품명': '국밥_순대국밥', '탄수화물(g)': '10.38', '지방': '2.28', '단백질': '3.17', '나트륨': '126.000', '총칼로리': '75.00', '기준량': '100g'}, {'식품명': '국밥_콩나물', '탄수화물(g)': '10.93', '지방': '0.24', '단백질': '1.45', '나트륨': '172.000', '총칼로리': '52.00', '기준량': '100g'}]
   